In [1]:
# ── CELL 1: Imports ───────────────────────────────────────────────────────────
import re
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm
import circadapt
from circadapt.error import ModelCrashed, ModelNotStable

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

MMHG_TO_PA  = 133.322
LPM_TO_M3PS = 1.0 / 60000.0

In [2]:
# ── CELL 2: Load and standardize data ────────────────────────────────────────
DATA_PATH = Path("SAS Database_multiple populations_for Amee Sangani.xlsx")

def standardize_column_name(col):
    col = str(col).strip()
    for old, new in [("%","pct"),("/","_per_"),("+","_plus"),("-","_minus"), ("(","_"),(")","\"")]:
        col = col.replace(old, new)
    col = re.sub(r"[^\w\s\.]+", "", col)
    col = re.sub(r"\s+", "_", col)
    col = re.sub(r"_+", "_", col)
    return col.strip("_").lower()

raw_df = pd.read_excel(DATA_PATH, sheet_name=0)
raw_df.columns = [standardize_column_name(c) for c in raw_df.columns]
print(f"Loaded: {raw_df.shape[0]} rows × {raw_df.shape[1]} columns")

Loaded: 350 rows × 110 columns


In [3]:
# ── CELL 3: Build model_df ────────────────────────────────────────────────────
df = raw_df.copy()

df["patient_id"] = (df["study"].astype(str).str.strip() + "_" +
                    df["subjectid"].astype(str).str.strip())

df["co_final"] = df["co_swan"]
mask = df["co_final"].isna() & df["co"].notna()
df.loc[mask, "co_final"] = df.loc[mask, "co"]

df["condition_label"] = df["condition_description"].astype(str).str.strip().str.lower()
df["is_rest"] = df["condition_label"].str.contains("rest", na=False)

model_df = pd.DataFrame({
    "patient_id" : df["patient_id"],
    "condition"  : df["condition_label"],
    "is_rest"    : df["is_rest"],
    "hr_bpm"     : df["hr"],
    "co_l_min"   : df["co_final"],
    "map_mmhg"   : df["map"],
    "ef_pct"     : df["ef"],
    "edv_ml"     : df["edv"],
    "esv_ml"     : df["esv"],
    "pcwp_mmhg"  : df["pcwp"],
    "sbp_mmhg"   : df["sbp"],
    "dbp_mmhg"   : df["dbp"],
    "rap_mmhg"   : df["rap"],
    "mpap_mmhg"  : df["mpap"],
    "age"        : df["age"],
    "sex"        : df["sex"],
    "bsa"        : df["bsa"],
})

# Force numeric types — raw Excel uses '.' as missing placeholder
numeric_cols = ["hr_bpm","co_l_min","map_mmhg","sbp_mmhg","dbp_mmhg",
                "edv_ml","esv_ml","ef_pct","pcwp_mmhg","rap_mmhg","mpap_mmhg",
                "age","bsa"]
for col in numeric_cols:
    if col in model_df.columns:
        model_df[col] = pd.to_numeric(model_df[col], errors="coerce")

model_df["usable_for_fit"] = (
    model_df["hr_bpm"].notna()   &
    model_df["co_l_min"].notna() &
    model_df["map_mmhg"].notna() &
    model_df["ef_pct"].notna()   &
    model_df["edv_ml"].notna()   &
    model_df["esv_ml"].notna()
)

print(f"model_df: {len(model_df)} rows, {model_df['patient_id'].nunique()} patients")
print(f"Usable for fit: {model_df['usable_for_fit'].sum()}")

model_df: 350 rows, 44 patients
Usable for fit: 145


In [4]:
# ── CELL 4: Build clean rest_df ───────────────────────────────────────────────
rest_df = model_df[model_df["is_rest"] & model_df["usable_for_fit"]].copy()

rest_df = rest_df[
    (rest_df["edv_ml"] > 0) & (rest_df["esv_ml"] > 0) &
    (rest_df["edv_ml"] > rest_df["esv_ml"]) &
    (rest_df["hr_bpm"]   >= 30)  & (rest_df["hr_bpm"]   <= 200) &
    (rest_df["co_l_min"] >= 1.0) & (rest_df["co_l_min"] <= 12.0) &
    (rest_df["map_mmhg"] >= 40)  & (rest_df["map_mmhg"] <= 180) &
    (rest_df["edv_ml"]   <= 500) & (rest_df["esv_ml"]   <= 400) &
    (rest_df["ef_pct"]   >= 10)  & (rest_df["ef_pct"]   <= 95)
].copy()

print(f"rest_df: {len(rest_df)} rows, {rest_df['patient_id'].nunique()} patients")
print(rest_df["condition"].value_counts())

rest_df: 77 rows, 30 patients
condition
supine rest                       24
upright rest                      20
invasive_normoxia_supine_rest      5
invasive_normoxia_upright_rest     5
invasive_rest_fio21                5
invasive_rest_fio17                5
invasive_rest_fio15                5
invasive_upright_rest_fio12        5
invasive_rest_fio12                3
Name: count, dtype: int64


In [21]:
# ── CELL 5 FIXED: clean base state snapshot ───────────────────────────────────
# Create fresh model, run to stable, snapshot BEFORE any pinning
_BASE_MODEL = circadapt.VanOsta2024()
_BASE_MODEL.run(stable=True)

# Explicitly ensure PFC is inactive before snapshotting
_BASE_MODEL['PFC']['is_active'] = False
_BASE_STATE = _BASE_MODEL.model_export()

# Verify the snapshot is clean
m_test = circadapt.VanOsta2024()
m_test.model_import(_BASE_STATE)
m_test, ok, err = run_model_safe(m_test, n_beats=5)
print(f"Snapshot verify: ok={ok}  err={err}")

# Store default theta from clean snapshot
_DEFAULT_THETA = get_theta(_BASE_MODEL)
print(f"Default theta: {_DEFAULT_THETA}")

# Fix reset_model to create a FRESH object every time
# (avoids shared-state corruption entirely)
def reset_model():
    """Always return a fresh model loaded from clean snapshot."""
    m = circadapt.VanOsta2024()
    m.model_import(_BASE_STATE)
    return m

# Fix simulate_with_theta to use fresh model
def simulate_with_theta(row, theta, n_beats=5):
    m = reset_model()                    # fresh object every call
    m = set_theta(m, theta)
    m = set_pinned_inputs(m,
                          hr_bpm=row.get("hr_bpm"),
                          co_l_min=row.get("co_l_min"))
    m, ok, err = run_model_safe(m, n_beats=n_beats)
    if not ok:
        return None, False, err
    try:
        return extract_outputs(m), True, None
    except Exception as e:
        return None, False, f"extract_failed: {e}"

def simulate_row(row, theta, n_beats=5):
    return simulate_with_theta(row, theta, n_beats)

def simulate_row_final(row, theta):
    return simulate_with_theta(row, theta, n_beats=10)

# Timing test
import time
row0 = rest_df_clean.iloc[2]
t0 = time.time()
s, ok, err = simulate_with_theta(row0, _DEFAULT_THETA)
print(f"\nok={ok}  err={err}  time={time.time()-t0:.2f}s")
print(s)

Default theta: {'Am_LV': 0.00966859, 'Sf_pas_LV': 731.24545453, 'Sf_act_LV': 120000.0, 'Vw_LV': 7.35720515e-05, 'p0_PVR': 12200.0}
ok=True  err=None  time=0.18s
{'map_mmhg': 91.18952395696795, 'sbp_mmhg': 123.51007351171305, 'dbp_mmhg': 65.88112587804265, 'co_l_min': 5.836008776104574, 'edv_ml': 136.89673797014783, 'esv_ml': 46.79569593510646, 'ef_pct': 65.81679254818263, 'pcwp_mmhg': 10.337935153814424}


In [22]:
# ── CELL 6: Fitting functions ─────────────────────────────────────────────────
THETA_BOUNDS = {
    "Am_LV"     : (0.002,  0.030),
    "Sf_pas_LV" : (200,    8000),
    "Sf_act_LV" : (5000,   400000),
    "Vw_LV"     : (0.00005, 0.0006),
    "p0_PVR"    : (3000,   40000),
}

INPUT_BOUNDS = {
    "hr_bpm"   : (30,  200),
    "co_l_min" : (1.0, 12.0),
    "map_mmhg" : (40,  180),
}

def clip_theta(theta):
    return {k: float(np.clip(v, *THETA_BOUNDS[k])) if k in THETA_BOUNDS else v
            for k, v in theta.items()}

def inputs_are_valid(row):
    for col, (lo, hi) in INPUT_BOUNDS.items():
        val = row.get(col)
        if pd.isna(val): return False, f"{col} is NaN"
        if not (lo <= float(val) <= hi): return False, f"{col}={val:.2f} outside [{lo},{hi}]"
    return True, None

def fit_patient(row, n_outer=4, n_inner=6, verbose=False):
    obs = {k: row.get(k) for k in
           ["map_mmhg","edv_ml","esv_ml","ef_pct","pcwp_mmhg","co_l_min"]}

    # Use stored default theta — never call reset_model() here
    # reset_model() is only called inside simulate_with_theta
    theta = _DEFAULT_THETA.copy()

    has = {k: pd.notna(obs[k]) and float(obs[k]) > 0
           for k in ["map_mmhg","edv_ml","esv_ml","ef_pct"]}
    has["pcwp"] = (pd.notna(obs.get("pcwp_mmhg")) and
                   float(obs.get("pcwp_mmhg") or 0) >= 0)

    def safe_sim():
        s, ok, err = simulate_with_theta(row, theta)
        return s if ok else None

    def do_pass(param, target, exp):
        for _ in range(n_inner):
            s = safe_sim()
            if s is None: return False
            if s[target] > 0:
                theta[param] *= (float(obs[target]) / s[target]) ** exp
            # clip in place
            if param in THETA_BOUNDS:
                lo, hi = THETA_BOUNDS[param]
                theta[param] = float(np.clip(theta[param], lo, hi))
        return True

    # Pass A: p0_PVR → MAP
    if has["map_mmhg"]:
        do_pass("p0_PVR", "map_mmhg", 1.0)

    # Outer loop
    loss_history = []
    for outer in range(n_outer):
        if has["edv_ml"]:                        do_pass("Am_LV",     "edv_ml", 0.5)
        if has["ef_pct"]:                         do_pass("Sf_act_LV", "ef_pct", 0.5)
        if has["esv_ml"] and has["edv_ml"]:       do_pass("Vw_LV",     "esv_ml", 0.3)

        s = safe_sim()
        if s:
            loss = sum(
                ((s[k] - float(obs[k])) / max(float(obs[k]), 1)) ** 2
                for k in ["map_mmhg","edv_ml","esv_ml","ef_pct"] if has.get(k)
            )
            loss_history.append(loss)
            if verbose:
                print(f"  outer {outer+1}/{n_outer}  loss={loss:.4f}"
                      f"  MAP={s['map_mmhg']:.1f}  EDV={s['edv_ml']:.1f}"
                      f"  ESV={s['esv_ml']:.1f}  EF={s['ef_pct']:.1f}")

    # Pass E: Sf_pas_LV → PCWP
    if has["pcwp"] and float(obs.get("pcwp_mmhg") or 0) > 2:
        do_pass("Sf_pas_LV", "pcwp_mmhg", 0.4)

    # Final simulation
    sim_final, ok, err = simulate_row_final(row, theta)
    if not ok or sim_final is None:
        return theta, {}, False, loss_history

    errors = [abs(sim_final[k] - float(obs[k])) / max(float(obs[k]), 1)
              for k in ["map_mmhg","edv_ml","ef_pct"] if has.get(k)]
    converged = all(e < 0.05 for e in errors) if errors else False

    return theta, sim_final, converged, loss_history

# Smoke test
print("Smoke test on first row...")
theta_t, sim_t, conv_t, lh_t = fit_patient(
    rest_df_clean.iloc[0], n_outer=2, n_inner=3, verbose=True
)
print(f"converged={conv_t}  sim={sim_t}")

Smoke test on first row...
  outer 1/2  loss=0.0071  MAP=100.0  EDV=174.1  ESV=82.1  EF=52.8
  outer 2/2  loss=0.0065  MAP=100.0  EDV=174.5  ESV=82.5  EF=52.7
converged=True  sim={'map_mmhg': 100.13746527191913, 'sbp_mmhg': 131.6264257573815, 'dbp_mmhg': 74.21535158641493, 'co_l_min': 5.723454595103497, 'edv_ml': 173.1907433326409, 'esv_ml': 82.57663305495282, 'ef_pct': 52.3204118961768, 'pcwp_mmhg': 10.232535357865354}


In [18]:
# ── CELL 7: Smoke test on one patient ────────────────────────────────────────
theta_t, sim_t, conv_t, lh_t = fit_patient(rest_df.iloc[0], n_outer=4, n_inner=8, verbose=True)
print(f"\nConverged: {conv_t}")
if sim_t:
    row0 = rest_df.iloc[0]
    print(f"\n{'Metric':<14} {'Observed':>10} {'Simulated':>10} {'Error':>10}")
    print("-" * 48)
    for k in ["map_mmhg","co_l_min","edv_ml","esv_ml","ef_pct","pcwp_mmhg"]:
        o = row0.get(k, float("nan"))
        s = sim_t.get(k, float("nan"))
        print(f"  {k:<12} {float(o):>10.2f} {float(s):>10.2f} {float(s)-float(o):>10.2f}")

  outer 1/4  loss=0.0077  MAP=99.8  EDV=173.8  ESV=81.9  EF=52.9
  outer 2/4  loss=0.0076  MAP=99.8  EDV=173.9  ESV=81.9  EF=52.9
  outer 3/4  loss=0.0074  MAP=99.7  EDV=174.1  ESV=82.0  EF=52.9
  outer 4/4  loss=0.0074  MAP=99.7  EDV=174.1  ESV=82.0  EF=52.9

Converged: True

Metric           Observed  Simulated      Error
------------------------------------------------
  map_mmhg          99.67     100.09       0.42
  co_l_min           5.65       5.74       0.08
  edv_ml           175.33     172.95      -2.38
  esv_ml            89.67      82.25      -7.42
  ef_pct            53.00      52.44      -0.56
  pcwp_mmhg           nan       9.85        nan


In [9]:
# ── CELL 8: Cohort-wide fitting loop ─────────────────────────────────────────
results, skipped, failed = [], [], []

print(f"Fitting {len(rest_df)} rows across {rest_df['patient_id'].nunique()} patients...\n")

for idx, row in tqdm(rest_df.iterrows(), total=len(rest_df)):
    pid = row["patient_id"]
    valid, reason = inputs_are_valid(row)
    if not valid:
        skipped.append({"patient_id": pid, "reason": reason}); continue

    theta, sim_fit, converged, loss_hist = fit_patient(row, n_outer=2, n_inner=3)

    if not sim_fit:
        failed.append({"patient_id": pid, "hr": row["hr_bpm"], "co": row["co_l_min"]}); continue

    results.append({
        "patient_id": pid, "condition": row["condition"], "converged": converged,
        "final_loss": loss_hist[-1] if loss_hist else float("nan"),
        "obs_map": row["map_mmhg"],   "sim_map": sim_fit["map_mmhg"],
        "obs_co":  row["co_l_min"],   "sim_co":  sim_fit["co_l_min"],
        "obs_edv": row["edv_ml"],     "sim_edv": sim_fit["edv_ml"],
        "obs_esv": row["esv_ml"],     "sim_esv": sim_fit["esv_ml"],
        "obs_ef":  row["ef_pct"],     "sim_ef":  sim_fit["ef_pct"],
        "obs_pcwp":row["pcwp_mmhg"],  "sim_pcwp":sim_fit["pcwp_mmhg"],
        "theta_Am_LV":     theta["Am_LV"],
        "theta_Sf_pas_LV": theta["Sf_pas_LV"],
        "theta_Sf_act_LV": theta["Sf_act_LV"],
        "theta_Vw_LV":     theta["Vw_LV"],
        "theta_p0_PVR":    theta["p0_PVR"],
    })

    # checkpoint every 10 rows
    if len(results) % 10 == 0:
        pd.DataFrame(results).to_csv("fitting_results_checkpoint.csv", index=False)
        tqdm.write(f"  checkpoint saved ({len(results)} rows done)")

results_df = pd.DataFrame(results)
results_df.to_csv("fitting_results_rest.csv", index=False)
print(f"\nFitted: {len(results_df)}  |  Converged: {results_df['converged'].sum()}"
      f"  |  Skipped: {len(skipped)}  |  Failed: {len(failed)}")
if failed:
    display(pd.DataFrame(failed))

Fitting 77 rows across 30 patients...



  0%|          | 0/77 [00:00<?, ?it/s]

  checkpoint saved (10 rows done)
  checkpoint saved (20 rows done)
  checkpoint saved (30 rows done)
  checkpoint saved (40 rows done)
  checkpoint saved (50 rows done)

Fitted: 54  |  Converged: 4  |  Skipped: 0  |  Failed: 23


,patient_id,hr,co
0,"21-4354, hypoxia_10",75.000000,5.741773
1,"21-4354, hypoxia_10",72.000000,8.988442
2,"21-4354, hypoxia_10",73.000000,10.211547
3,"21-4354, hypoxia_10",72.000000,11.119218
4,"21-4354, hypoxia_10",80.000000,10.112690
5,"21-4354, hypoxia_10",90.000000,6.826218
6,"17-1141, LVAD_102",62.000000,10.780000
7,"17-1141, LVAD_102",70.500000,2.711429
8,"17-1141, LVAD_104",90.200000,5.800000
9,"17-1141, LVAD_104",90.500000,5.616667


In [10]:
# Cell 10 — RMSE summary
def rmse(a, b):
    mask = pd.notna(a) & pd.notna(b)
    return float(np.sqrt(np.mean((b[mask] - a[mask]) ** 2))) if mask.sum() else float("nan")

print(f"\n{'Metric':<16} {'N':>4} {'RMSE':>8}")
print("-" * 32)
for label, oc, sc in [
    ("MAP (mmHg)",  "obs_map",  "sim_map"),
    ("CO (L/min)",  "obs_co",   "sim_co"),
    ("EDV (mL)",    "obs_edv",  "sim_edv"),
    ("ESV (mL)",    "obs_esv",  "sim_esv"),
    ("EF (%)",      "obs_ef",   "sim_ef"),
    ("PCWP (mmHg)", "obs_pcwp", "sim_pcwp"),
]:
    n = (pd.notna(results_df[oc]) & pd.notna(results_df[sc])).sum()
    r = rmse(results_df[oc], results_df[sc])
    print(f"  {label:<14} {n:>4} {r:>8.2f}")

print(f"\nConvergence: {results_df['converged'].mean()*100:.1f}%")
print(f"\nBy condition:")
print(results_df.groupby("condition")["converged"].agg(["sum","count"]))


Metric              N     RMSE
--------------------------------
  MAP (mmHg)       54     4.54
  CO (L/min)       54     0.48
  EDV (mL)         54    30.35
  ESV (mL)         54    19.44
  EF (%)           54     8.53
  PCWP (mmHg)      19    29.17

Convergence: 7.4%

By condition:
                                sum  count
condition                                 
invasive_normoxia_supine_rest     0      5
invasive_normoxia_upright_rest    0      4
invasive_rest_fio12               0      2
invasive_rest_fio15               0      4
invasive_rest_fio17               1      4
invasive_rest_fio21               0      4
invasive_upright_rest_fio12       0      4
supine rest                       1     15
upright rest                      2     12


In [19]:
# Exclude LVAD patients and rerun with better settings
rest_df_clean = rest_df[~rest_df["patient_id"].str.contains("LVAD", case=False)].copy()
print(f"Rows after excluding LVAD: {len(rest_df_clean)} across {rest_df_clean['patient_id'].nunique()} patients")
print(rest_df_clean["condition"].value_counts())

Rows after excluding LVAD: 60 across 21 patients
condition
supine rest                       15
upright rest                      12
invasive_normoxia_supine_rest      5
invasive_normoxia_upright_rest     5
invasive_rest_fio21                5
invasive_rest_fio17                5
invasive_rest_fio15                5
invasive_upright_rest_fio12        5
invasive_rest_fio12                3
Name: count, dtype: int64


##Run 2

In [14]:
# Re-initialize the base model snapshot cleanly
_BASE_MODEL = circadapt.VanOsta2024()
_BASE_MODEL.run(stable=True)
_BASE_STATE = _BASE_MODEL.model_export()
print("Base model re-initialized")

# Now test again
row = rest_df_clean.iloc[0]
theta0 = get_theta(reset_model())
sim, ok, err = simulate_row_final(row, theta0)
print(f"ok={ok}  err={err}")
print(sim)

Base model re-initialized
ok=True  err=None
{'map_mmhg': 91.82632390609761, 'sbp_mmhg': 124.15212021411386, 'dbp_mmhg': 66.36305796427706, 'co_l_min': 5.662649882635424, 'edv_ml': 137.19988787692594, 'esv_ml': 47.05393915437965, 'ef_pct': 65.70409795335328, 'pcwp_mmhg': 10.38638547256929}


In [20]:
# ── REFIT on clean cohort ─────────────────────────────────────────────────────
results2, skipped2, failed2 = [], [], []

print(f"Fitting {len(rest_df_clean)} rows across {rest_df_clean['patient_id'].nunique()} patients...\n")

for idx, row in tqdm(rest_df_clean.iterrows(), total=len(rest_df_clean)):
    pid = row["patient_id"]
    valid, reason = inputs_are_valid(row)
    if not valid:
        skipped2.append({"patient_id": pid, "reason": reason}); continue

    theta, sim_fit, converged, loss_hist = fit_patient(row, n_outer=4, n_inner=6)

    if not sim_fit:
        failed2.append({"patient_id": pid, "hr": row["hr_bpm"], "co": row["co_l_min"]}); continue

    results2.append({
        "patient_id": pid, "condition": row["condition"], "converged": converged,
        "final_loss": loss_hist[-1] if loss_hist else float("nan"),
        "obs_map": row["map_mmhg"],   "sim_map": sim_fit["map_mmhg"],
        "obs_co":  row["co_l_min"],   "sim_co":  sim_fit["co_l_min"],
        "obs_edv": row["edv_ml"],     "sim_edv": sim_fit["edv_ml"],
        "obs_esv": row["esv_ml"],     "sim_esv": sim_fit["esv_ml"],
        "obs_ef":  row["ef_pct"],     "sim_ef":  sim_fit["ef_pct"],
        "obs_pcwp":row["pcwp_mmhg"],  "sim_pcwp":sim_fit["pcwp_mmhg"],
        "theta_Am_LV":     theta["Am_LV"],
        "theta_Sf_pas_LV": theta["Sf_pas_LV"],
        "theta_Sf_act_LV": theta["Sf_act_LV"],
        "theta_Vw_LV":     theta["Vw_LV"],
        "theta_p0_PVR":    theta["p0_PVR"],
    })

    if len(results2) % 10 == 0:
        pd.DataFrame(results2).to_csv("fitting_results_clean_checkpoint.csv", index=False)
        tqdm.write(f"  checkpoint saved ({len(results2)} rows done)")

results2_df = pd.DataFrame(results2)
results2_df.to_csv("fitting_results_clean.csv", index=False)

print(f"\nFitted: {len(results2_df)}  |  Converged: {results2_df['converged'].sum()}"
      f"  |  Skipped: {len(skipped2)}  |  Failed: {len(failed2)}")
if failed2:
    print("\nFailed rows:")
    display(pd.DataFrame(failed2))

Fitting 60 rows across 21 patients...



  0%|          | 0/60 [00:00<?, ?it/s]


Fitted: 5  |  Converged: 2  |  Skipped: 0  |  Failed: 55

Failed rows:


,patient_id,hr,co
0,"17-1141, healthy control_6",54.00,3.941200
1,"17-1141, healthy control_6",68.00,5.961429
2,"17-1141, healthy control_7",60.00,3.855900
3,"17-1141, healthy control_7",61.00,5.408750
4,"17-1141, healthy control_9",70.00,7.811549
5,"17-1141, healthy control_9",74.00,7.969000
6,"17-1141, healthy control_10",58.00,5.010580
7,"17-1141, healthy control_10",66.00,4.496250
8,"17-1141, healthy control_12",59.00,4.366880
9,"17-1141, healthy control_12",62.40,4.871429


In [23]:
# ── REFIT LOOP with group-aware CO bounds ─────────────────────────────────────
results2, skipped2, failed2 = [], [], []

# Wider CO bound for hypoxia patients who genuinely have high cardiac output
def inputs_are_valid_adaptive(row):
    hr  = row.get("hr_bpm")
    co  = row.get("co_l_min")
    map_ = row.get("map_mmhg")

    if pd.isna(hr)  or not (30  <= float(hr)  <= 200): return False, f"hr={hr}"
    if pd.isna(map_) or not (40  <= float(map_) <= 180): return False, f"map={map_}"
    if pd.isna(co):  return False, "co is NaN"

    # hypoxia patients can have CO up to 14 L/min physiologically
    pid = str(row.get("patient_id",""))
    co_max = 14.0 if "hypoxia" in pid.lower() else 12.0
    if not (1.0 <= float(co) <= co_max): return False, f"co={co:.2f} outside [1,{co_max}]"

    return True, None

print(f"Fitting {len(rest_df_clean)} rows across {rest_df_clean['patient_id'].nunique()} patients...\n")

for idx, row in tqdm(rest_df_clean.iterrows(), total=len(rest_df_clean)):
    pid = row["patient_id"]

    valid, reason = inputs_are_valid_adaptive(row)
    if not valid:
        skipped2.append({"patient_id": pid, "reason": reason}); continue

    theta, sim_fit, converged, loss_hist = fit_patient(row, n_outer=4, n_inner=6)

    if not sim_fit:
        # Log with reason — check if it's a known hard case
        is_hfref   = "hfref" in pid.lower()
        is_hypoxia = "hypoxia" in pid.lower()
        failed2.append({
            "patient_id": pid,
            "group": "HFrEF" if is_hfref else ("hypoxia" if is_hypoxia else "other"),
            "hr": row["hr_bpm"], "co": row["co_l_min"],
            "ef": row["ef_pct"], "edv": row["edv_ml"],
        }); continue

    results2.append({
        "patient_id": pid, "condition": row["condition"], "converged": converged,
        "final_loss": loss_hist[-1] if loss_hist else float("nan"),
        "obs_map":  row["map_mmhg"],  "sim_map":  sim_fit["map_mmhg"],
        "obs_co":   row["co_l_min"],  "sim_co":   sim_fit["co_l_min"],
        "obs_edv":  row["edv_ml"],    "sim_edv":  sim_fit["edv_ml"],
        "obs_esv":  row["esv_ml"],    "sim_esv":  sim_fit["esv_ml"],
        "obs_ef":   row["ef_pct"],    "sim_ef":   sim_fit["ef_pct"],
        "obs_pcwp": row["pcwp_mmhg"], "sim_pcwp": sim_fit["pcwp_mmhg"],
        "theta_Am_LV":     theta["Am_LV"],
        "theta_Sf_pas_LV": theta["Sf_pas_LV"],
        "theta_Sf_act_LV": theta["Sf_act_LV"],
        "theta_Vw_LV":     theta["Vw_LV"],
        "theta_p0_PVR":    theta["p0_PVR"],
    })

    if len(results2) % 10 == 0:
        pd.DataFrame(results2).to_csv("fitting_results_clean_checkpoint.csv", index=False)
        tqdm.write(f"  checkpoint saved ({len(results2)} rows done)")

results2_df = pd.DataFrame(results2)
results2_df.to_csv("fitting_results_clean.csv", index=False)

print(f"\nFitted:    {len(results2_df)}")
print(f"Converged: {results2_df['converged'].sum() if len(results2_df) else 0}")
print(f"Skipped:   {len(skipped2)}")
print(f"Failed:    {len(failed2)}")

if failed2:
    fdf = pd.DataFrame(failed2)
    print("\nFailed by group:")
    print(fdf["group"].value_counts())
    print("\nFailed EF distribution:")
    print(fdf["ef"].describe())

Fitting 60 rows across 21 patients...



  0%|          | 0/60 [00:00<?, ?it/s]


Fitted:    5
Converged: 2
Skipped:   0
Failed:    55

Failed by group:
group
hypoxia    33
HFrEF      12
other      10
Name: count, dtype: int64

Failed EF distribution:
count    55.000000
mean     49.516046
std      16.707121
min      14.333333
25%      35.000000
50%      50.800000
75%      61.866667
max      87.000000
Name: ef, dtype: float64


In [24]:
import inspect
print(inspect.getsource(fit_patient))

def fit_patient(row, n_outer=4, n_inner=6, verbose=False):
    obs = {k: row.get(k) for k in
           ["map_mmhg","edv_ml","esv_ml","ef_pct","pcwp_mmhg","co_l_min"]}

    # Use stored default theta — never call reset_model() here
    # reset_model() is only called inside simulate_with_theta
    theta = _DEFAULT_THETA.copy()

    has = {k: pd.notna(obs[k]) and float(obs[k]) > 0
           for k in ["map_mmhg","edv_ml","esv_ml","ef_pct"]}
    has["pcwp"] = (pd.notna(obs.get("pcwp_mmhg")) and
                   float(obs.get("pcwp_mmhg") or 0) >= 0)

    def safe_sim():
        s, ok, err = simulate_with_theta(row, theta)
        return s if ok else None

    def do_pass(param, target, exp):
        for _ in range(n_inner):
            s = safe_sim()
            if s is None: return False
            if s[target] > 0:
                theta[param] *= (float(obs[target]) / s[target]) ** exp
            # clip in place
            if param in THETA_BOUNDS:
                lo, hi = THETA_BO

In [25]:
print("_DEFAULT_THETA" in dir())
print("simulate_with_theta" in dir())

True
True


In [26]:
row_test = rest_df_clean.iloc[2]  # healthy control_6
print(f"patient: {row_test['patient_id']}, EF={row_test['ef_pct']}, CO={row_test['co_l_min']}")
theta_t, sim_t, conv_t, lh_t = fit_patient(row_test, n_outer=1, n_inner=2, verbose=True)
print(f"sim_t: {sim_t}")

patient: 17-1141, healthy control_3, EF=57.142857142857146, CO=6.9399999999999995
sim_t: {}


In [27]:
# Exact failure diagnosis
row_test = rest_df_clean.iloc[2]
theta_test = _DEFAULT_THETA.copy()

# Test simulate_with_theta directly
s, ok, err = simulate_with_theta(row_test, theta_test, n_beats=5)
print(f"Default sim: ok={ok}  err={err}")
print(f"outputs: {s}")

# Now run one pass and test again
if ok:
    # manually do pass A
    if s["map_mmhg"] > 0:
        theta_test["p0_PVR"] *= float(row_test["map_mmhg"]) / s["map_mmhg"]
        lo, hi = THETA_BOUNDS["p0_PVR"]
        theta_test["p0_PVR"] = float(np.clip(theta_test["p0_PVR"], lo, hi))
    print(f"\nAfter p0_PVR update: {theta_test['p0_PVR']:.1f}")
    
    s2, ok2, err2 = simulate_with_theta(row_test, theta_test, n_beats=5)
    print(f"After pass A: ok={ok2}  err={err2}")
    print(f"outputs: {s2}")

Default sim: ok=False  err=ModelCrashed
outputs: None


In [28]:
row_test = rest_df_clean.iloc[2]
print(f"HR={row_test['hr_bpm']}  CO={row_test['co_l_min']}  MAP={row_test['map_mmhg']}")
print(f"t_cycle = {60.0 / float(row_test['hr_bpm']):.4f} s")
print(f"q0 = {float(row_test['co_l_min']) * LPM_TO_M3PS:.6f} m3/s")

# Test with NO pinning at all
m = reset_model()
m, ok, err = run_model_safe(m, n_beats=5)
print(f"\nNo pinning: ok={ok}  err={err}")

# Test with only CO pinned
m = reset_model()
m = set_pinned_inputs(m, hr_bpm=None, co_l_min=row_test["co_l_min"])
m, ok, err = run_model_safe(m, n_beats=5)
print(f"CO only pinned: ok={ok}  err={err}")

# Test with only HR pinned
m = reset_model()
m = set_pinned_inputs(m, hr_bpm=row_test["hr_bpm"], co_l_min=None)
m, ok, err = run_model_safe(m, n_beats=5)
print(f"HR only pinned: ok={ok}  err={err}")

# Test with both pinned
m = reset_model()
m = set_pinned_inputs(m, hr_bpm=row_test["hr_bpm"], co_l_min=row_test["co_l_min"])
m, ok, err = run_model_safe(m, n_beats=5)
print(f"Both pinned: ok={ok}  err={err}")

HR=63.0  CO=6.9399999999999995  MAP=96.66666666666667
t_cycle = 0.9524 s
q0 = 0.000116 m3/s

No pinning: ok=False  err=ModelCrashed
CO only pinned: ok=False  err=ModelCrashed
HR only pinned: ok=False  err=ModelCrashed
Both pinned: ok=False  err=ModelCrashed


In [29]:
# DIAGNOSTIC: does a brand new VanOsta2024() work?
m_fresh = circadapt.VanOsta2024()
m_fresh, ok, err = run_model_safe(m_fresh, n_beats=5)
print(f"Fresh model: ok={ok}  err={err}")

if ok:
    row_test = rest_df_clean.iloc[2]
    m_fresh2 = circadapt.VanOsta2024()
    m_fresh2 = set_pinned_inputs(m_fresh2,
                                  hr_bpm=row_test["hr_bpm"],
                                  co_l_min=row_test["co_l_min"])
    m_fresh2, ok2, err2 = run_model_safe(m_fresh2, n_beats=5)
    print(f"Fresh + pinned: ok={ok2}  err={err2}")
    if ok2:
        print(extract_outputs(m_fresh2))

Fresh model: ok=True  err=None
Fresh + pinned: ok=True  err=None
{'map_mmhg': 90.1927809854554, 'sbp_mmhg': 125.98596516272633, 'dbp_mmhg': 62.34560998149537, 'co_l_min': 7.471390063627172, 'edv_ml': 149.4185814599651, 'esv_ml': 46.732219320579496, 'ef_pct': 68.7239573124305, 'pcwp_mmhg': 13.723301513559834}
